In [4]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Population_UNData.csv")
df.head()

,Country,Year,Population
0,China,1950,555000000
1,China,1955,612000000
2,China,1960,669000000
3,China,1965,726000000
4,China,1970,783000000


In [8]:
india = df[df["Country"]=="India"]
china = df[df["Country"]=="China"]

fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=china["Year"],
        y=china["Population"]/1000000,
        mode="lines",
        name="China",
        line=dict(color="lightgray", width=4)
    )
)

fig1.add_trace(
    go.Scatter(
        x=india["Year"],
        y=india["Population"]/1000000,
        mode="lines",
        name="India",
        line=dict(color="red", width=6)
    )
)

fig1.update_layout(
    title="India supera a China como el país más poblado del mundo",
    xaxis_title="Año",
    yaxis_title="Población (millones)",
    template="simple_white"
)

fig1.show()

In [9]:
ultimo_año = df["Year"].max()

top10 = (
    df[df["Year"] == ultimo_año]
    .sort_values("Population", ascending=False)
)

colores = ["red"] + ["lightgray"]*(len(top10)-1)

fig2 = go.Figure()

fig2.add_trace(
    go.Bar(
        x=top10["Population"]/1000000,
        y=top10["Country"],
        orientation="h",
        marker_color=colores
    )
)

fig2.update_layout(
    title=f"Países más poblados en {ultimo_año}",
    xaxis_title="Millones de habitantes",
    yaxis_title="",
    template="simple_white"
)

fig2.show()

In [10]:
primer_año = df["Year"].min()

inicio = df[df["Year"]==primer_año][["Country","Population"]]
final = df[df["Year"]==ultimo_año][["Country","Population"]]

crecimiento = inicio.merge(
    final,
    on="Country",
    suffixes=("_inicio","_final")
)

crecimiento["Growth"] = (
    (
        crecimiento["Population_final"]
        -
        crecimiento["Population_inicio"]
    )
    /
    crecimiento["Population_inicio"]
)*100

crecimiento = crecimiento.sort_values(
    "Growth",
    ascending=False
)

fig3 = go.Figure()

fig3.add_trace(
    go.Bar(
        x=crecimiento["Country"],
        y=crecimiento["Growth"],
        marker_color="steelblue"
    )
)

fig3.update_layout(
    title="Crecimiento porcentual de la población",
    xaxis_title="País",
    yaxis_title="% Crecimiento",
    template="simple_white"
)

fig3.show()

In [16]:
participacion = (
    df[df["Year"]==ultimo_año]
    .copy()
)

participacion["Participacion"] = (
    participacion["Population"]
    /
    participacion["Population"].sum()
)*100

fig4 = go.Figure()

fig4.add_trace(
    go.Pie(
        labels=participacion["Country"],
        values=participacion["Participacion"],
        hole=0.45
    )
)

fig4.update_layout(
    title="Participación de la población total"
)

fig4.show()

In [14]:
fig_dashboard = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "India vs China",
        "Ranking Actual",
        "Crecimiento %",
        "Participación Mundial"
    ),
    specs=[
        [{"type":"scatter"},{"type":"bar"}],
        [{"type":"bar"},{"type":"pie"}]
    ]
)

# --------------------
# Gráfico 1
# --------------------

fig_dashboard.add_trace(
    go.Scatter(
        x=china["Year"],
        y=china["Population"]/1000000,
        mode="lines",
        name="China",
        line=dict(color="lightgray")
    ),
    row=1,
    col=1
)

fig_dashboard.add_trace(
    go.Scatter(
        x=india["Year"],
        y=india["Population"]/1000000,
        mode="lines",
        name="India",
        line=dict(color="red")
    ),
    row=1,
    col=1
)

# --------------------
# Gráfico 2
# --------------------

fig_dashboard.add_trace(
    go.Bar(
        x=top10["Population"]/1000000,
        y=top10["Country"],
        orientation="h",
        name="Ranking"
    ),
    row=1,
    col=2
)

# --------------------
# Gráfico 3
# --------------------

fig_dashboard.add_trace(
    go.Bar(
        x=crecimiento["Country"],
        y=crecimiento["Growth"],
        name="Growth"
    ),
    row=2,
    col=1
)

# --------------------
# Gráfico 4
# --------------------

fig_dashboard.add_trace(
    go.Bar(
        labels=participacion["Country"],
        values=participacion["Participacion"],
        name="Participación"
    ),
    row=2,
    col=2
)

fig_dashboard.update_layout(
    title="Dashboard: Transformación Demográfica Mundial",
    height=900,
    width=1400,
    template="simple_white"
)

fig_dashboard.show()